In [ ]:
using DifferentialEquations, Plots

In [ ]:
# all constants
g_na = 120.0
g_k = 36.0
g_l = 0.3
c_m = 1.0
I_ext = 10.0
E_na = 50.0
E_k = -77.0
E_l = -54.4

In [ ]:
# rate funtions

# --- Potassium Gating (n) ---
alpha_n(V) = abs(V + 55.0) < 1e-6 ? 0.1 : 0.01 * (V + 55.0) / (1.0 - exp(-(V + 55.0) / 10.0))
beta_n(V) = 0.125 * exp(-(V + 65.0) / 80.0)


# --- Sodium Activation (m) ---
alpha_m(V) = abs(V + 40.0) < 1e-6 ? 1.0 : 0.1 * (V + 40.0) / (1.0 - exp(-(V + 40.0) / 10.0))
beta_m(V) = 4.0 * exp(-(V + 65.0) / 18.0)

# --- Sodium Inactivation (h) ---
alpha_h(V) = 0.07 * exp(-(V + 65.0) / 20.0)
beta_h(V) = 1.0 / (1.0 + exp(-(V + 35.0) / 10.0))

## ode Hudgkin-huxley (HH)  model

In [ ]:
function HH_model!(du, u_0, p, t)

    V, m, h, n = u_0
    g_na, g_k, g_l, c_m, I_ext, E_na, E_k, E_l = p




    du[1] = 1 / c_m * (I_ext - g_na * m^3 * h * (V - E_na) - g_k * n^4 * (V - E_k) - g_l * (V - E_l))
    du[2] = alpha_m(V) * (1 - m) - beta_m(V) * (m)
    du[3] = alpha_h(V) * (1 - h) - beta_h(V) * (h)
    du[4] = alpha_n(V) * (1 - n) - beta_n(V) * (n)
end

In [ ]:
u_0 = [-65.0, 0.05, 0.6, 0.317]
tspan = (0.0, 30.0)
p = [g_na, g_k, g_l, c_m, I_ext, E_na, E_k, E_l]

In [ ]:
#defining the ode problem
prob = ODEProblem(HH_model!, u_0, tspan, p)

# solution of ode

In [ ]:

sol = solve(prob, Rosenbrock23(), reltol=1e-6, abstol=1e-6,adaptive=false,dt = 0.01)


In [ ]:
println(length(sol[1, :]))
println(summary(sol[1, :]))

In [ ]:
t = sol.t

In [ ]:
V=sol[1, :]
m = sol[2, :]
h= sol[3, :]
n=sol[4, :]
println(length(V))
println(length(m))
println(length(h))
println(length(n))


In [ ]:
V

In [ ]:
plot(title = " Neural Spiking" ,t,V,xlabel = "Time (s)",ylabel = "volatage",size(400,300),background_color=:black)

In [ ]:
p = plot(title ="n_gate",  t,m,xlabel = "Time (s)",ylabel = "m",label = "m",background_color=:black)
plot!(t,h,xlabel = "Time (s)",ylabel = "h",label = "h")
plot!(t,n,xlabel = "Time (s)",ylabel = "n",label = "n")


In [ ]:
using DataFrames

In [ ]:
df = DataFrame( Time = t, V = V, m = m , h= h,n=n )

In [ ]:
using CSV

In [ ]:
CSV.write("synthetic_data.csv",df)